vgg与经典卷积神经网络类似，由一系列卷积层组成，后面再加上用于空间下采样的最大汇聚层。

实现VGG块

该函数有三个参数，分别对应于卷积层的数量num_convs、输入通道的数量in_channels 和输出通道的数量out_channels

In [1]:
import torch
from torch import nn

def vgg_block(num_convs, in_channels, out_channels):
    layers = []
    for _ in range(num_convs):
        layers.append(nn.Conv2d(in_channels, out_channels,
                                kernel_size=3, padding=1))
        layers.append(nn.ReLU())
        in_channels = out_channels
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2))
    return nn.Sequential(*layers)

与AlexNet、LeNet一样，VGG网络可以分为两部分：第一部分主要由卷积层和汇聚层组成，第二部分由全连接层组成。

conv_arch变量指定了每个VGG块里卷积层个数和输出通道数，全连接模块则与AlexNet中的相同。

In [2]:
conv_arch = ((1, 64), (1, 128), (2, 256), (2, 512), (2, 512))

实现vgg-11，该网络有5个卷积块，前两个块有一个卷积层，后三个有两个卷积层，第一个模块64个输出通道，每个后续模块通道数量翻倍，直至512。共8个卷积层和3个全连接层。

In [3]:
def vgg(conv_arch):
    conv_blks = []
    in_channels = 1
    # 卷积层部分
    for (num_convs, out_channels) in conv_arch:
        conv_blks.append(vgg_block(num_convs, in_channels, out_channels))
        in_channels = out_channels

    return nn.Sequential(
        *conv_blks, nn.Flatten(),
        # 全连接层部分
        nn.Linear(out_channels * 7 * 7, 4096), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(4096, 10))

net = vgg(conv_arch)

构建高度和宽度为224的单通道数据样本，观察每个层输出的形状。

In [4]:
X = torch.randn(size=(1, 1, 224, 224))
for blk in net:
    X = blk(X)
    print(blk.__class__.__name__,'output shape:\t',X.shape)

Sequential output shape:	 torch.Size([1, 64, 112, 112])
Sequential output shape:	 torch.Size([1, 128, 56, 56])
Sequential output shape:	 torch.Size([1, 256, 28, 28])
Sequential output shape:	 torch.Size([1, 512, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
Flatten output shape:	 torch.Size([1, 25088])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 10])


训练模型

由于VGG-11比AlexNet计算量更大，因此我们构建了一个通道数较少的网络，来训练Fashion-MNIST数据集

In [5]:
ratio = 4
small_conv_arch = [(pair[0], pair[1] // ratio) for pair in conv_arch]
net = vgg(small_conv_arch)

In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

# 超参数
lr, num_epochs, batch_size = 0.05, 10, 128

# 选择设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on", device)

# 数据预处理
# Fashion-MNIST 原图是 28x28，这里放大到 224x224 以适配 VGG
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 下载数据集
train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

# 数据加载器
train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_iter = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# 计算准确率
def evaluate_accuracy_gpu(net, data_iter, device):
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return correct / total

# 训练函数
def train_ch6(net, train_iter, test_iter, num_epochs, lr, device):
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        net.train()
        metric_loss = 0.0
        metric_correct = 0
        metric_total = 0
        start = time.time()

        for X, y in train_iter:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()

            metric_loss += l.item() * X.shape[0]
            metric_correct += (y_hat.argmax(dim=1) == y).sum().item()
            metric_total += y.numel()

        train_loss = metric_loss / metric_total
        train_acc = metric_correct / metric_total
        test_acc = evaluate_accuracy_gpu(net, test_iter, device)

        print(f"epoch {epoch + 1}, "
              f"train loss {train_loss:.4f}, "
              f"train acc {train_acc:.4f}, "
              f"test acc {test_acc:.4f}, "
              f"time {time.time() - start:.1f} sec")

# 开始训练
train_ch6(net, train_iter, test_iter, num_epochs, lr, device)

training on cuda


100%|██████████| 26.4M/26.4M [00:01<00:00, 17.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 269kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.96MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.7MB/s]


epoch 1, train loss 2.3027, train acc 0.0987, test acc 0.1001, time 23.3 sec
epoch 2, train loss 2.3014, train acc 0.1245, test acc 0.3323, time 21.7 sec
epoch 3, train loss 1.0758, train acc 0.6187, test acc 0.8187, time 21.4 sec
epoch 4, train loss 0.4487, train acc 0.8324, test acc 0.8502, time 21.4 sec
epoch 5, train loss 0.3513, train acc 0.8705, test acc 0.8811, time 21.6 sec
epoch 6, train loss 0.3036, train acc 0.8872, test acc 0.8910, time 21.6 sec
epoch 7, train loss 0.2723, train acc 0.8999, test acc 0.8992, time 21.4 sec
epoch 8, train loss 0.2459, train acc 0.9091, test acc 0.9034, time 21.4 sec
epoch 9, train loss 0.2255, train acc 0.9162, test acc 0.9043, time 21.5 sec
epoch 10, train loss 0.2075, train acc 0.9233, test acc 0.9077, time 21.5 sec


## 练习

1. 打印层的尺寸时，我们只看到8个结果，而不是11个结果。剩余的3层信息去哪了？


在VGG网络中，打印结果较少是因为多个卷积层被封装在VGG block 中，循环只遍历块而不是单独的卷积层，因此部分层被隐藏在块内部。

后三个块中各有两个卷积层，因此被隐藏

1. 与AlexNet相比，VGG的计算要慢得多，而且它还需要更多的显存。分析出现这种情况的原因。

网络更深，层数更多

使用多个 3×3 卷积替代大卷积核，增加计算次数

前期特征图分辨率较高，占用更多显存

通道数增加较快，特征图更大

全连接层参数量巨大

1. 尝试将Fashion-MNIST数据集图像的高度和宽度从224改为96。这对实验有什么影响？

定义适配96×96输入的VGG

In [24]:
small_conv_arch = ((1, 16), (1, 32), (2, 64), (2, 128), (2, 128))

def vgg_96(conv_arch):
    layers = []
    in_channels = 1
    for (num_convs, out_channels) in conv_arch:
        layers.append(vgg_block(num_convs, in_channels, out_channels))
        in_channels = out_channels

    net = nn.Sequential(
        *layers,
        nn.Flatten(),
        nn.Linear(out_channels * 3 * 3, 512),   # 注意这里是 3*3
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 10)
    )
    return net

net = vgg_96(small_conv_arch)

加载数据集

In [9]:
batch_size = 128

transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor()
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_iter = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

训练

In [10]:
lr, num_epochs = 0.05, 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on", device)

train_ch6(net, train_iter, test_iter, num_epochs, lr, device)

training on cuda
epoch 1, train loss 2.3028, train acc 0.0979, test acc 0.1000, time 9.3 sec
epoch 2, train loss 2.3028, train acc 0.0974, test acc 0.1000, time 8.7 sec
epoch 3, train loss 2.3027, train acc 0.0996, test acc 0.1000, time 8.6 sec
epoch 4, train loss 2.3027, train acc 0.0998, test acc 0.1000, time 9.3 sec
epoch 5, train loss 2.3027, train acc 0.1001, test acc 0.1000, time 8.9 sec
epoch 6, train loss 2.3025, train acc 0.1021, test acc 0.1000, time 9.2 sec
epoch 7, train loss 2.3011, train acc 0.1220, test acc 0.2702, time 8.7 sec
epoch 8, train loss 1.4551, train acc 0.4725, test acc 0.7033, time 9.2 sec
epoch 9, train loss 0.6228, train acc 0.7625, test acc 0.8077, time 8.9 sec
epoch 10, train loss 0.4860, train acc 0.8160, test acc 0.8370, time 8.9 sec


模型仍然可以正常训练，但需要相应修改VGG全连接层的输入维度，因为经过5次最大池化后，特征图尺寸由7×7变为3×3。

实验中可以观察到，输入尺寸减小后，训练速度明显提升，显存占用显著降低，模型更适合在普通计算环境下运行。与此同时，由于输入分辨率下降，模型提取到的空间细节减少，测试精度略有下降。

1. 请参考VGG论文 :cite:`Simonyan.Zisserman.2014`中的表1构建其他常见模型，如VGG-16或VGG-19。

VGG-16

前两个块有两个卷积层，后三个块有三个卷积层

In [11]:
conv_arch_vgg16 = ((2, 64), (2, 128), (3, 256), (3, 512), (3, 512))

In [38]:
def vgg(conv_arch, in_channels=1, num_classes=10):
    layers = []
    for (num_convs, out_channels) in conv_arch:
        layers.append(vgg_block(num_convs, in_channels, out_channels))
        in_channels = out_channels

    return nn.Sequential(
        *layers,
        nn.Flatten(),
        nn.Linear(in_channels * 3 * 3, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes)

    )

net = vgg(conv_arch_vgg16)

def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

net.apply(init_weights)

Sequential(
  (0): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (1): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (2): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (3): Sequential(
   

In [39]:
lr, num_epochs = 0.01, 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on", device)

train_ch6(net, train_iter, test_iter, num_epochs, lr, device)

training on cuda
epoch 1, train loss 2.2943, train acc 0.2326, test acc 0.4924, time 21.7 sec
epoch 2, train loss 1.4554, train acc 0.4978, test acc 0.7310, time 21.6 sec
epoch 3, train loss 0.7411, train acc 0.7239, test acc 0.7777, time 21.6 sec
epoch 4, train loss 0.5933, train acc 0.7830, test acc 0.8160, time 21.6 sec
epoch 5, train loss 0.5131, train acc 0.8131, test acc 0.8366, time 21.6 sec
epoch 6, train loss 0.4620, train acc 0.8344, test acc 0.8399, time 21.7 sec
epoch 7, train loss 0.4291, train acc 0.8441, test acc 0.8536, time 21.7 sec
epoch 8, train loss 0.4028, train acc 0.8542, test acc 0.8617, time 21.7 sec
epoch 9, train loss 0.3787, train acc 0.8633, test acc 0.8671, time 21.7 sec
epoch 10, train loss 0.3613, train acc 0.8699, test acc 0.8720, time 21.6 sec
